# 02 — Cleaning QA & Data Quality Validation

This notebook follows Notebook 01 and performs **structured QA checks** on the cleaned dataset.

It does not change business logic. It documents quality, nulls, ranges, and saves an optional CLI mirror file for reproducibility comparison.

---
## 1) Setup

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'cleaned_olist_dataset.csv'
CLI_MIRROR_PATH = PROJECT_ROOT / 'data' / 'processed' / 'cleaned_olist_dataset_cli.csv'

df = pd.read_csv(DATA_PATH)
print(f'Loaded cleaned dataset: {len(df):,} rows × {len(df.columns)} columns')

Loaded cleaned dataset: 110,197 rows × 48 columns


---
## 2) Schema and Type Health

In [2]:
dtype_view = pd.DataFrame({
    'column': df.columns,
    'dtype': [str(t) for t in df.dtypes],
    'non_null': [df[c].notna().sum() for c in df.columns],
    'nulls': [df[c].isna().sum() for c in df.columns],
    'null_pct': [df[c].isna().mean() * 100 for c in df.columns],
}).sort_values('null_pct', ascending=False)
dtype_view.head(20)

,column,dtype,non_null,nulls,null_pct
18,product_name_length,float64,108660,1537,1.394775
20,product_photos_qty,float64,108660,1537,1.394775
19,product_description_length,float64,108660,1537,1.394775
35,review_answer_timestamp,str,109370,827,0.750474
34,review_creation_date,str,109370,827,0.750474
33,review_score,float64,109370,827,0.750474
32,max_installments,float64,110106,91,0.082579
31,primary_payment_type,str,110106,91,0.082579
24,product_width_cm,float64,110179,18,0.016334
46,product_volume_cm3,float64,110179,18,0.016334


---
## 3) Key Integrity Checks

In [3]:
checks = {}
checks['row_count'] = len(df)
checks['col_count'] = len(df.columns)
checks['unique_orders'] = df['order_id'].nunique()
checks['delivered_only_ratio'] = (df['order_status'].eq('delivered').mean() * 100) if 'order_status' in df.columns else np.nan
checks['review_score_missing_pct'] = df['review_score'].isna().mean() * 100
checks['delivery_delay_missing_pct'] = df['delivery_delay_days'].isna().mean() * 100
checks['on_time_rate_pct'] = df['is_on_time'].astype(float).mean() * 100

pd.Series(checks)

row_count                     110197.000000
col_count                         48.000000
unique_orders                  96478.000000
delivered_only_ratio             100.000000
review_score_missing_pct           0.750474
delivery_delay_missing_pct         0.007260
on_time_rate_pct                  93.400909
dtype: float64

---
## 4) Numeric Sanity Bands (Spot gross outliers / impossible values)

In [4]:
numeric_cols = [c for c in [
    'price', 'freight_value', 'total_item_value', 'total_payment_value',
    'delivery_delay_days', 'actual_delivery_days', 'approval_lag_hours'
] if c in df.columns]

df[numeric_cols].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T

,count,mean,std,min,1%,5%,50%,95%,99%,max
price,110197.0,119.980563,182.299446,0.85,9.9900,17.000000,74.900000,349.000000,887.076000,6735.000000
freight_value,110197.0,19.948598,15.698136,0.00,4.3500,7.780000,16.260000,45.090000,83.830000,409.680000
total_item_value,110197.0,139.929161,189.319151,6.08,20.8696,30.060000,92.130000,375.260000,920.608000,6929.310000
total_payment_value,110194.0,179.466763,271.338343,9.59,22.9486,33.476500,114.345000,527.793500,1230.000000,13664.080000
delivery_delay_days,110189.0,-12.029041,10.158194,-147.00,-36.0000,-27.000000,-13.000000,3.000000,18.000000,188.000000
actual_delivery_days,110189.0,12.007342,9.451153,0.00,1.0000,2.000000,10.000000,29.000000,45.000000,209.000000
approval_lag_hours,110182.0,10.518197,20.986374,0.00,0.0000,0.144722,0.350278,48.688014,90.276147,741.443611


---
## 5) Save Optional CLI Mirror Copy

This keeps compatibility with existing repo artifacts used elsewhere.

In [5]:
df.to_csv(CLI_MIRROR_PATH, index=False)
print('Saved:', CLI_MIRROR_PATH)

Saved: /Users/hemanth10etii/Coding/DVACap/data/processed/cleaned_olist_dataset_cli.csv


---
## Notebook Output

- Input: `data/processed/cleaned_olist_dataset.csv`
- Optional mirror output: `data/processed/cleaned_olist_dataset_cli.csv`

Next notebook: **`03_eda.ipynb`**